# AdamW из финальной точки MM с экспоненциальной регуляризацией

Ноутбук проверяет следующие гипотезы

- является ли финальная точка MM стационарной для точной регуляризованной цели
- может ли AdamW продолжить уменьшать ту же самую цель
- уменьшается ли только train loss или также улучшаются официальные test метрики
- насколько велики разрывы касательной оценки первого слагаемого и ELBO после дополнительного MM шага

AdamW запускается с `weight_decay=0`
Экспоненциально линейный штраф добавляется явно и полностью совпадает со штрафом `mm_regularized`


## 1 Настройка окружения

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'configs').is_dir():
    PROJECT_DIR = Path.cwd().resolve().parent
if not (PROJECT_DIR / 'configs').is_dir():
    raise FileNotFoundError('Запустите ноутбук из корня catsbench или из папки notebooks')

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.methods.dlight_sb import DLightSB
from src.methods.mm_regularized import _exponential_log_penalty

print('project', PROJECT_DIR)
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

## 2 Параметры эксперимента

Для точного сравнения AdamW оптимизирует ту же регуляризованную цель что и MM

При необходимости укажите `CHECKPOINT_PATH` вручную


In [ ]:
EXPERIMENT = 'mm_regularized/benchmark_hd/d16_g002'
DIMENSION = 16
SEED = 42

CHECKPOINT_PATH = None
PREVIOUS_MM_CHECKPOINT = None

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ADAMW_BATCH_SIZE = 16_384
DIAGNOSTIC_BATCH_SIZE = 16_384
PROBE_STEPS = 100
FULL_STEPS = 2_000
LEARNING_RATES = (1e-2, 3e-3, 1e-3)
EVAL_EVERY = 25
GRAD_CLIP_NORM = None

RUN_MM_PROBE = True
RUN_OFFICIAL_TESTS = False

RESULTS_DIR = PROJECT_DIR / 'logs' / 'adamw_from_mm' / f'd{DIMENSION}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('device', DEVICE)
print('results', RESULTS_DIR)

## 3 Поиск MM чекпоинта

In [ ]:
def find_latest_checkpoint() -> Path:
    pattern = f'mm_regularized/benchmark_hd/d{DIMENSION}_g002/**/checkpoints/last.ckpt'
    candidates = list((PROJECT_DIR / 'logs' / 'runs').glob(pattern))
    if not candidates:
        raise FileNotFoundError(
            'Чекпоинт не найден автоматически. Укажите CHECKPOINT_PATH вручную'
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)

checkpoint_path = (
    Path(CHECKPOINT_PATH).expanduser().resolve()
    if CHECKPOINT_PATH is not None
    else find_latest_checkpoint()
)
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)

print('MM checkpoint', checkpoint_path)

## 4 Данные и конфигурация

Диагностический батч фиксируется один раз
AdamW получает новые выборки из того же генератора данных


In [ ]:
overrides = [
    f'experiment={EXPERIMENT}',
    f'data.batch_size={ADAMW_BATCH_SIZE}',
    'data.num_workers=0',
]
with initialize_config_dir(version_base='1.1', config_dir=str(PROJECT_DIR / 'configs')):
    cfg = compose(config_name='config.yaml', overrides=overrides)

datamodule = instantiate(cfg.data)
datamodule.prepare_data()
datamodule.setup('fit')

train_loader = datamodule.train_dataloader()
train_iterator = iter(train_loader)

def next_train_batch():
    global train_iterator
    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)
    return tuple(t.to(DEVICE, non_blocking=True) for t in batch)

diagnostic_x0 = datamodule.benchmark.sample_input(DIAGNOSTIC_BATCH_SIZE).to(DEVICE)
diagnostic_x1 = datamodule.benchmark.sample_target(DIAGNOSTIC_BATCH_SIZE).to(DEVICE)
diagnostic_batch = (diagnostic_x0, diagnostic_x1)

print(OmegaConf.to_yaml(cfg.method))
print('diagnostic shapes', diagnostic_x0.shape, diagnostic_x1.shape)

## 5 Загрузка параметров MM в дифференцируемую модель

Класс MM отключает градиенты поэтому для AdamW создаётся обычный `DLightSB` с тем же prior и теми же параметрами


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
checkpoint_state = checkpoint['state_dict']

def build_adam_model() -> DLightSB:
    mm_template = instantiate(cfg.method)
    mm_template.load_state_dict(checkpoint_state, strict=True)
    model = DLightSB(
        prior=copy.deepcopy(mm_template.prior),
        dim=mm_template.hparams.dim,
        num_categories=mm_template.hparams.num_categories,
        num_potentials=mm_template.hparams.num_potentials,
        num_timesteps=mm_template.hparams.num_timesteps,
        optimizer=None,
        scheduler=None,
        distr_init=mm_template.hparams.distr_init,
        entropy_lambda=0,
        entropy_warmup_steps=mm_template.hparams.entropy_warmup_steps,
        sample_prob=mm_template.hparams.sample_prob,
        tau=mm_template.hparams.tau,
    )
    model.load_state_dict(mm_template.state_dict(), strict=True)
    model.requires_grad_(True)
    return model.to(DEVICE)

regularization_lambda = float(cfg.method.regularization_lambda)
base_model = build_adam_model()
initial_state = copy.deepcopy(base_model.state_dict())

print('lambda', regularization_lambda)
print('parameters', sum(parameter.numel() for parameter in base_model.parameters()))

## 6 Точная цель и нормы градиентов

Измеряются градиенты data loss и полной регуляризованной цели


In [ ]:
def objective_parts(model: DLightSB, batch):
    x0, x1 = batch
    data_loss, info = model.loss(x0, x1)
    regularization = regularization_lambda * (
        _exponential_log_penalty(model.log_alpha).sum()
        + _exponential_log_penalty(model.log_cp_cores).sum()
    )
    total = data_loss + regularization
    return total, data_loss, regularization, info

def gradient_statistics(model: DLightSB, batch):
    result = {}
    for name in ('data', 'total'):
        model.zero_grad(set_to_none=True)
        total, data_loss, regularization, _ = objective_parts(model, batch)
        selected = data_loss if name == 'data' else total
        selected.backward()
        gradients = [parameter.grad.detach().flatten() for parameter in model.parameters()]
        vector = torch.cat(gradients)
        result[f'{name}_grad_norm'] = vector.norm().item()
        result[f'{name}_grad_rms'] = vector.square().mean().sqrt().item()
        result[f'{name}_grad_max'] = vector.abs().max().item()
    model.zero_grad(set_to_none=True)
    result.update(
        total=total.item(),
        data_loss=data_loss.item(),
        regularization=regularization.item(),
    )
    return result

initial_gradient = gradient_statistics(base_model, diagnostic_batch)
pd.Series(initial_gradient, name='MM endpoint')

## 7 Разрывы двух частей мажоранты

Для двух наборов параметров вычисляются

\[
\Delta_+=\frac1N\sum_i\left[\exp\left(\delta_i\right)-1-\delta_i\right]
\]

\[
\delta_i=\log c_i^{\mathrm{new}}-\log c_i^{\mathrm{old}}
\]

и KL разрыв ELBO по ответственностям компонент


In [ ]:
@torch.no_grad()
def target_log_responsibilities(model: DLightSB, x1: torch.Tensor):
    x1 = x1.flatten(1)
    logits = model.log_alpha.expand(len(x1), -1).clone()
    for dimension in range(model.hparams.dim):
        logits += model.log_cp_cores[dimension, x1[:, dimension]]
    return logits.log_softmax(dim=1)

@torch.no_grad()
def bound_gap_statistics(old_model: DLightSB, new_model: DLightSB, batch):
    x0, x1 = batch
    old_log_c = old_model.get_log_c(x0)
    new_log_c = new_model.get_log_c(x0)
    delta = new_log_c - old_log_c
    source_gap = torch.expm1(delta) - delta

    old_log_gamma = target_log_responsibilities(old_model, x1)
    new_log_gamma = target_log_responsibilities(new_model, x1)
    target_gap = (old_log_gamma.exp() * (old_log_gamma - new_log_gamma)).sum(dim=1)

    def summary(prefix, values):
        quantiles = torch.quantile(
            values.float(),
            torch.tensor([0.5, 0.95, 0.99], device=values.device),
        )
        return {
            f'{prefix}_mean': values.mean().item(),
            f'{prefix}_median': quantiles[0].item(),
            f'{prefix}_p95': quantiles[1].item(),
            f'{prefix}_p99': quantiles[2].item(),
            f'{prefix}_max': values.max().item(),
        }

    return {
        **summary('source_gap', source_gap),
        **summary('elbo_gap', target_gap),
        'total_gap_mean': (source_gap.mean() + target_gap.mean()).item(),
        'abs_log_c_change_mean': delta.abs().mean().item(),
        'abs_log_c_change_max': delta.abs().max().item(),
    }

## 8 Дополнительный MM шаг из точки плато

Этот тест показывает может ли текущая MM карта продолжить уменьшение цели на фиксированном диагностическом батче


In [ ]:
mm_probe_result = None
if RUN_MM_PROBE:
    mm_probe = instantiate(cfg.method).to(DEVICE)
    mm_probe.load_state_dict(checkpoint_state, strict=True)
    mm_probe._loaded_from_ckpt = True
    mm_before = build_adam_model()
    mm_info = mm_probe.mm_step(*diagnostic_batch)
    mm_after = build_adam_model()
    mm_after.load_state_dict(mm_probe.state_dict(), strict=True)
    mm_probe_result = {
        **{f'mm_{key}': value for key, value in mm_info.items()},
        **bound_gap_statistics(mm_before, mm_after, diagnostic_batch),
    }
    display(pd.Series(mm_probe_result, name='one extra MM step'))
else:
    print('MM probe отключён')

## 9 Необязательное сравнение с предыдущим MM чекпоинтом

Если сохранён предпоследний чекпоинт укажите `PREVIOUS_MM_CHECKPOINT` чтобы измерить реальные разрывы последнего обучающего MM шага


In [ ]:
previous_gap_result = None
if PREVIOUS_MM_CHECKPOINT is not None:
    previous_payload = torch.load(
        Path(PREVIOUS_MM_CHECKPOINT).expanduser().resolve(),
        map_location='cpu',
        weights_only=False,
    )
    previous_model = build_adam_model()
    previous_model.load_state_dict(previous_payload['state_dict'], strict=True)
    previous_gap_result = bound_gap_statistics(previous_model, base_model, diagnostic_batch)
    display(pd.Series(previous_gap_result, name='previous MM checkpoint to final MM checkpoint'))
else:
    print('PREVIOUS_MM_CHECKPOINT не задан')

## 10 AdamW probe для нескольких learning rate

Каждый запуск начинается из одного и того же MM чекпоинта
Лучший learning rate выбирается только по фиксированному диагностическому батчу


In [ ]:
def evaluate_model(model: DLightSB, batch):
    model.eval()
    with torch.no_grad():
        total, data_loss, regularization, info = objective_parts(model, batch)
    return {
        'total': total.item(),
        'data_loss': data_loss.item(),
        'regularization': regularization.item(),
        'log_c': info['log_c'].item(),
        'log_v': info['log_v'].item(),
        'posterior_eff_k': info['posterior_eff_k'].item(),
        'prior_eff_k': info['prior_eff_k'].item(),
    }

def train_adamw(learning_rate: float, steps: int, label: str):
    model = build_adam_model()
    optimizer = torch.optim.AdamW(
        [model.log_alpha, model.log_cp_cores],
        lr=learning_rate,
        betas=(0.95, 0.99),
        weight_decay=0.0,
    )
    history = []
    for step in range(steps + 1):
        if step == 0 or step % EVAL_EVERY == 0 or step == steps:
            row = evaluate_model(model, diagnostic_batch)
            history.append({'label': label, 'lr': learning_rate, 'step': step, **row})
        if step == steps:
            break

        model.train()
        optimizer.zero_grad(set_to_none=True)
        total, _, _, _ = objective_parts(model, next_train_batch())
        total.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        if not torch.isfinite(model.log_alpha).all() or not torch.isfinite(model.log_cp_cores).all():
            raise FloatingPointError(f'Нечисловые параметры при lr={learning_rate} step={step}')

    return model, pd.DataFrame(history)

probe_models = {}
probe_histories = []
for learning_rate in LEARNING_RATES:
    print('probe lr', learning_rate)
    model, history = train_adamw(learning_rate, PROBE_STEPS, 'probe')
    probe_models[learning_rate] = model
    probe_histories.append(history)

probe_history = pd.concat(probe_histories, ignore_index=True)
probe_summary = (
    probe_history.sort_values('step')
    .groupby('lr', as_index=False)
    .tail(1)
    .sort_values('total')
)
display(probe_summary)

## 11 Полный AdamW запуск

Полный запуск снова начинается из исходной точки MM с выбранным learning rate


In [ ]:
best_learning_rate = float(probe_summary.iloc[0]['lr'])
print('selected lr', best_learning_rate)

adam_model, adam_history = train_adamw(
    best_learning_rate,
    FULL_STEPS,
    'full',
)
final_gradient = gradient_statistics(adam_model, diagnostic_batch)
adam_gap_from_mm = bound_gap_statistics(base_model, adam_model, diagnostic_batch)

comparison = pd.DataFrame([initial_gradient, final_gradient], index=['MM endpoint', 'AdamW final'])
display(comparison)
display(pd.Series(adam_gap_from_mm, name='MM endpoint to AdamW final'))

## 12 Кривые оптимизации

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for learning_rate, group in probe_history.groupby('lr'):
    axes[0].plot(group['step'], group['total'], marker='o', label=f'lr={learning_rate:g}')
axes[0].set_title('AdamW probe')
axes[0].set_xlabel('step')
axes[0].set_ylabel('regularized objective')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(adam_history['step'], adam_history['data_loss'], label='data loss')
axes[1].plot(adam_history['step'], adam_history['total'], label='total')
axes[1].set_title('Full AdamW run')
axes[1].set_xlabel('step')
axes[1].legend()
axes[1].grid(alpha=0.25)

axes[2].plot(adam_history['step'], adam_history['regularization'], color='tab:orange')
axes[2].set_title('Exponential regularization')
axes[2].set_xlabel('step')
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 13 Сохранение AdamW чекпоинта

Создаётся копия MM чекпоинта с заменёнными параметрами модели
Исходный чекпоинт не изменяется


In [ ]:
adam_checkpoint_path = RESULTS_DIR / 'adamw_from_mm.ckpt'
adam_checkpoint = copy.deepcopy(checkpoint)
adam_state = adam_model.state_dict()
for key in ('log_alpha', 'log_cp_cores'):
    adam_checkpoint['state_dict'][key] = adam_state[key].detach().cpu()
adam_checkpoint['optimizer_states'] = []
adam_checkpoint['lr_schedulers'] = []
adam_checkpoint['adamw_from_mm'] = {
    'source_checkpoint': str(checkpoint_path),
    'learning_rate': best_learning_rate,
    'steps': FULL_STEPS,
    'weight_decay': 0.0,
    'regularization_lambda': regularization_lambda,
}
torch.save(adam_checkpoint, adam_checkpoint_path)

adam_history.to_csv(RESULTS_DIR / 'adamw_history.csv', index=False)
probe_history.to_csv(RESULTS_DIR / 'adamw_probe_history.csv', index=False)
comparison.to_csv(RESULTS_DIR / 'gradient_comparison.csv')

print('saved', adam_checkpoint_path)

## 14 Официальные test метрики до и после AdamW

Эта ячейка запускает существующий test workflow проекта для обоих чекпоинтов
Она измеряет Shape Score Trend Score conditional метрики и trajectory KL если они предусмотрены текущим callback

Перед запуском установите `RUN_OFFICIAL_TESTS=True`


In [ ]:
def run_official_test(name: str, path: Path):
    output_dir = RESULTS_DIR / f'test_{name}'
    command = [
        sys.executable,
        '-m',
        'src.run',
        f'experiment={EXPERIMENT}',
        'task_name=test',
        f'ckpt_path={path.resolve().as_posix()}',
        f'hydra.run.dir={output_dir.resolve().as_posix()}',
        f'seed={SEED}',
    ]
    print(' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    metric_files = list(output_dir.rglob('metrics.csv'))
    if not metric_files:
        return {'name': name, 'output_dir': str(output_dir)}
    frame = pd.read_csv(max(metric_files, key=lambda item: item.stat().st_mtime))
    test_columns = [column for column in frame.columns if column.startswith('test/')]
    populated = frame[test_columns].dropna(how='all')
    result = populated.iloc[-1].dropna().to_dict() if len(populated) else {}
    return {'name': name, **result, 'output_dir': str(output_dir)}

official_metrics = None
if RUN_OFFICIAL_TESTS:
    official_metrics = pd.DataFrame([
        run_official_test('mm', checkpoint_path),
        run_official_test('adamw', adam_checkpoint_path),
    ]).set_index('name')
    display(official_metrics)
    official_metrics.to_csv(RESULTS_DIR / 'official_test_metrics.csv')
else:
    print('Официальный test отключён')

## 15 Автоматическая интерпретация

Главное сравнение строится по одной и той же регуляризованной цели на фиксированном диагностическом батче


In [ ]:
objective_improvement = initial_gradient['total'] - final_gradient['total']
data_loss_improvement = initial_gradient['data_loss'] - final_gradient['data_loss']
gradient_ratio = (
    final_gradient['total_grad_rms']
    / max(initial_gradient['total_grad_rms'], np.finfo(float).tiny)
)

diagnosis = {
    'objective_improvement': objective_improvement,
    'data_loss_improvement': data_loss_improvement,
    'initial_total_grad_rms': initial_gradient['total_grad_rms'],
    'final_total_grad_rms': final_gradient['total_grad_rms'],
    'final_to_initial_grad_ratio': gradient_ratio,
    'adam_source_gap_mean': adam_gap_from_mm['source_gap_mean'],
    'adam_elbo_gap_mean': adam_gap_from_mm['elbo_gap_mean'],
}
display(pd.Series(diagnosis, name='diagnosis'))

if objective_improvement > 0 and initial_gradient['total_grad_rms'] > final_gradient['total_grad_rms']:
    print('AdamW продолжил оптимизацию той же цели. MM endpoint не был конечной стационарной точкой для этой диагностики')
elif abs(objective_improvement) <= 1e-6:
    print('AdamW не дал заметного улучшения. Вероятнее стационарная точка или слишком малый градиент')
else:
    print('Результат неоднозначен. Проверьте learning rate кривые и нормы градиентов')

if official_metrics is not None:
    print('Сопоставьте изменение objective с изменением test метрик в таблице выше')